In [ ]:
import pandas as pd
import ast
import re
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import networkx as nx
from rapidfuzz import process, fuzz
from collections import Counter
import ast
import pyarrow
from ingredient_parser import parse_ingredient

[nltk_data] Error loading averaged_perceptron_tagger_eng: <urlopen
[nltk_data]     error [SSL: CERTIFICATE_VERIFY_FAILED] certificate
[nltk_data]     verify failed: unable to get local issuer certificate
[nltk_data]     (_ssl.c:992)>


aliases

In [2]:
UNIT_ALIASES = {
    "t": "teaspoon", "t.": "teaspoon", "tsp": "teaspoon", "tsp.": "teaspoon",
    "teaspoon": "teaspoon", "teaspoons": "teaspoon",

    "T": "tablespoon", "T.": "tablespoon", "tbsp": "tablespoon", "tbsp.": "tablespoon",
    "tbs": "tablespoon", "tbs.": "tablespoon", "tablespoon": "tablespoon",
    "tablespoons": "tablespoon",

    "c": "cup", "c.": "cup", "cup": "cup", "cups": "cup",

    "oz": "ounce", "oz.": "ounce", "ounce": "ounce", "ounces": "ounce",

    "lb": "pound", "lb.": "pound", "lbs": "pound", "lbs.": "pound",
    "pound": "pound", "pounds": "pound",

    "g": "gram", "g.": "gram", "gram": "gram", "grams": "gram",

    "kg": "kilogram", "kg.": "kilogram", "kilogram": "kilogram",
    "kilograms": "kilogram",

    "ml": "milliliter", "ml.": "milliliter", "mL": "milliliter",
    "milliliter": "milliliter", "milliliters": "milliliter",

    "l": "liter", "L": "liter", "liter": "liter", "liters": "liter",

    "pinch": "pinch", "pinches": "pinch",
    "dash": "dash", "dashes": "dash",
    "clove": "clove", "cloves": "clove",
    "slice": "slice", "slices": "slice",
    "can": "can", "cans": "can",
    "package": "package", "packages": "package",
    "pkg": "package", "pkg.": "package",
}

BASE_PATH = "./data"

load state

In [3]:
recipes_df = pd.read_parquet(f"{BASE_PATH}/recipes_trimmed_keep_longest.parquet")
ingredient_matches = pd.read_parquet(f"{BASE_PATH}/accepted_ingredient_matches_v1.parquet")
ingredient_candidates = pd.read_parquet(f"{BASE_PATH}/ingredient_food_candidates_v1.parquet")

embeddings = np.load(f"{BASE_PATH}/recipe_embeddings_fp16.npy")
index = faiss.read_index(f"{BASE_PATH}/recipe_index-001.faiss")

In [4]:
recipes_df["ingredients"].iloc[0]

array(['1 c. firmly packed brown sugar', '1/2 c. evaporated milk',
       '1/2 tsp. vanilla', '1/2 c. broken nuts (pecans)',
       '2 Tbsp. butter or margarine',
       '3 1/2 c. bite size shredded rice biscuits'], dtype=object)

create safe copy

In [5]:
recipes = recipes_df.copy(deep=True)

new parsed column

In [13]:
recipes["ingredients_parsed"] = recipes["ingredients"]

flatten for analysis

In [14]:
all_ingredients = [
    ing
    for ing_list in recipes["ingredients_parsed"]
    if isinstance(ing_list, (list, np.ndarray))
    for ing in ing_list
]

In [18]:
len(all_ingredients), all_ingredients[:10]

(16173715,
 ['1 c. firmly packed brown sugar',
  '1/2 c. evaporated milk',
  '1/2 tsp. vanilla',
  '1/2 c. broken nuts (pecans)',
  '2 Tbsp. butter or margarine',
  '3 1/2 c. bite size shredded rice biscuits',
  '1 small jar chipped beef, cut up',
  '4 boned chicken breasts',
  '1 can cream of mushroom soup',
  '1 carton sour cream'])

Now testing a ingredient unit parser based on these raw text keys alone. This will not be the final product

In [19]:
def clean_unit_token(token):
    if token is None:
        return None

    token = token.strip()

    if token in ["T", "T.", "L"]:
        return token

    return token.lower().replace(".", "")


def extract_possible_unit(line):
    pattern = r"^\s*(\d+\s+\d+/\d+|\d+/\d+|\d+(?:\.\d+)?)\s+([A-Za-z.]+)"
    match = re.match(pattern, str(line))

    if not match:
        return None

    return clean_unit_token(match.group(2))


def find_unknown_units(ingredient_lines):
    possible_units = []

    for line in ingredient_lines:
        unit = extract_possible_unit(line)
        if unit:
            possible_units.append(unit)

    counts = Counter(possible_units)

    unknown = {
        unit: count
        for unit, count in counts.items()
        if unit not in UNIT_ALIASES
    }

    return dict(sorted(unknown.items(), key=lambda x: x[1], reverse=True))

In [22]:
unknown_units = find_unknown_units(all_ingredients)

list(unknown_units.items())[:20]

[('large', 319767),
 ('eggs', 191992),
 ('medium', 185730),
 ('small', 160638),
 ('egg', 142224),
 ('garlic', 111620),
 ('whole', 108879),
 ('to', 107269),
 ('green', 59922),
 ('onion', 59182),
 ('stick', 55116),
 ('x', 44830),
 ('red', 41653),
 ('bunch', 35770),
 ('bay', 34343),
 ('box', 31967),
 ('each', 29480),
 ('qt', 29478),
 ('chicken', 29098),
 ('lemon', 28711)]

In [23]:
len(unknown_units)

10175

In [31]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt')

[nltk_data] Error loading averaged_perceptron_tagger_eng: <urlopen
[nltk_data]     error [SSL: CERTIFICATE_VERIFY_FAILED] certificate
[nltk_data]     verify failed: unable to get local issuer certificate
[nltk_data]     (_ssl.c:992)>
[nltk_data] Error loading punkt: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:992)>


False

In [32]:
nltk.download('averaged_perceptron_tagger')

[nltk_data] Error loading averaged_perceptron_tagger: <urlopen error
[nltk_data]     [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify
[nltk_data]     failed: unable to get local issuer certificate
[nltk_data]     (_ssl.c:992)>


False

A good start, but I would like to run the remaining and sanity check the rest as well with an LLM solution

starting with a sample set

In [25]:
sample_ingredients = all_ingredients[:10000]

In [26]:
def parse_with_status(line):
    try:
        parsed = parse_ingredient(str(line))

        has_amount = bool(parsed.amount)
        has_name = bool(parsed.name)

        if has_amount and has_name:
            status = "parsed"
        elif has_name and not has_amount:
            status = "missing_amount"
        else:
            status = "failed"

        return {
            "ingredient_raw": line,
            "status": status,
            "parsed": parsed,
            "error": None
        }

    except Exception as e:
        return {
            "ingredient_raw": line,
            "status": "error",
            "parsed": None,
            "error": str(e)
        }

parse

In [27]:
parsed_results = [parse_with_status(x) for x in sample_ingredients]

audit table of parse

In [28]:
parse_audit = pd.DataFrame([
    {
        "ingredient_raw": r["ingredient_raw"],
        "status": r["status"],
        "parsed": str(r["parsed"]),
        "error": r["error"]
    }
    for r in parsed_results
])

explore

In [29]:
parse_audit["status"].value_counts()

status
error    10000
Name: count, dtype: int64

In [30]:
for x in sample_ingredients[:5]:
    try:
        print(parse_ingredient(str(x)))
    except Exception as e:
        print(type(e))
        print(e)
        break

<class 'LookupError'>

**********************************************************************
  Resource 'averaged_perceptron_tagger_eng' not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('averaged_perceptron_tagger_eng')

  For more information see: https://www.nltk.org/data.html

  Attempted to load 'taggers/averaged_perceptron_tagger_eng/'

  Searched in:
    - '/Users/kadinwilkins/nltk_data'
    - '/Users/kadinwilkins/recipe_env/nltk_data'
    - '/Users/kadinwilkins/recipe_env/share/nltk_data'
    - '/Users/kadinwilkins/recipe_env/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************

